In [1]:
import pandas as pd
import itertools

edges = pd.read_csv(
    "/content/email-Eu-core.txt",
    sep=r"\s+",
    header=None,
    names=["source", "target"]
)

departments = pd.read_csv(
    "/content/email-Eu-core-department-labels.txt",
    sep=r"\s+",
    header=None,
    names=["node", "department"]
)

print("Email connections:")
print(edges.head())

print("\nDepartment labels:")
print(departments.head())

Email connections:
   source  target
0       0       1
1       2       3
2       2       4
3       5       6
4       5       7

Department labels:
   node  department
0     0           1
1     1           1
2     2          21
3     3          21
4     4          21


In [2]:
print("Total email connections:", len(edges))
print("Total people:", len(departments))
print("Total departments:", departments["department"].nunique())

Total email connections: 25571
Total people: 1005
Total departments: 42


In [3]:
# Create a copy of original edges
clean_edges = edges.copy()

# Count self-loops before removal
self_loops = (clean_edges["source"] == clean_edges["target"]).sum()

# Remove self-loops
clean_edges = clean_edges[
    clean_edges["source"] != clean_edges["target"]
]

# Count duplicate edges before removal
duplicates = clean_edges.duplicated().sum()

# Remove duplicate edges
clean_edges = clean_edges.drop_duplicates()

# Reset index
clean_edges = clean_edges.reset_index(drop=True)

print("Original edges:", len(edges))
print("Self-loops removed:", self_loops)
print("Duplicate edges removed:", duplicates)
print("Final clean edges:", len(clean_edges))

Original edges: 25571
Self-loops removed: 642
Duplicate edges removed: 0
Final clean edges: 24929


In [4]:
print("Remaining self-loops:",
      (clean_edges["source"] == clean_edges["target"]).sum())

print("Remaining duplicates:",
      clean_edges.duplicated().sum())

print("Final number of edges:", len(clean_edges))

Remaining self-loops: 0
Remaining duplicates: 0
Final number of edges: 24929


In [5]:
clean_edges.to_csv("clean_edges.csv", index=False)

departments.to_csv("department_labels.csv", index=False)

In [6]:
R = 3309

pair_number = R % 190

num_random_networks = 20 + (R % 31)

swap_multiplier = 5 + (R % 6)

random_seed = R

pairs = list(itertools.combinations(range(20), 2))

i, j = pairs[pair_number]

print("R:", R)
print("Pair number:", pair_number)
print("Pair positions:", i, j)
print("Random networks:", num_random_networks)
print("Swap multiplier:", swap_multiplier)
print("Random seed:", random_seed)

R: 3309
Pair number: 79
Pair positions: 4 14
Random networks: 43
Swap multiplier: 8
Random seed: 3309


In [7]:
# Add sender's department
edges_with_departments = clean_edges.merge(
    departments,
    left_on="source",
    right_on="node"
)

edges_with_departments = edges_with_departments.rename(
    columns={"department": "source_department"}
)

edges_with_departments = edges_with_departments.drop(
    columns=["node"]
)


# Add receiver's department
edges_with_departments = edges_with_departments.merge(
    departments,
    left_on="target",
    right_on="node"
)

edges_with_departments = edges_with_departments.rename(
    columns={"department": "target_department"}
)

edges_with_departments = edges_with_departments.drop(
    columns=["node"]
)


# Display results
print("Total rows:", len(edges_with_departments))
print("\nFirst 10 rows:")
print(edges_with_departments.head(10))

Total rows: 24929

First 10 rows:
   source  target  source_department  target_department
0       0       1                  1                  1
1       2       3                 21                 21
2       2       4                 21                 21
3       5       6                 25                 25
4       5       7                 25                 14
5       8       9                 14                 14
6      10      11                  9                 14
7      12      13                 14                 26
8      12      14                 14                  4
9      15      16                 17                 34


In [8]:
internal_edges = edges_with_departments[
    edges_with_departments["source_department"] ==
    edges_with_departments["target_department"]
]

print("Total internal edges:", len(internal_edges))

print("\nFirst 10 internal edges:")
print(internal_edges.head(10))

Total internal edges: 8645

First 10 internal edges:
    source  target  source_department  target_department
0        0       1                  1                  1
1        2       3                 21                 21
2        2       4                 21                 21
3        5       6                 25                 25
5        8       9                 14                 14
10      17      18                  1                  1
11      12      19                 14                 14
12      20      21                  9                  9
13      20      22                  9                  9
14      23      24                 11                 11


In [9]:
# Count internal edges for each department
internal_counts = (
    internal_edges.groupby("source_department")
    .size()
)

# Create a list of all 42 departments
all_departments = pd.DataFrame({
    "department": sorted(departments["department"].unique())
})

# Attach internal edge counts
department_counts = all_departments.merge(
    internal_counts.rename("internal_edges"),
    left_on="department",
    right_index=True,
    how="left"
)

# Replace missing counts with zero
department_counts["internal_edges"] = (
    department_counts["internal_edges"]
    .fillna(0)
    .astype(int)
)

print(department_counts)

    department  internal_edges
0            0             426
1            1             502
2            2              43
3            3              46
4            4            1167
5            5             147
6            6              21
7            7             679
8            8             134
9            9             146
10          10             472
11          11             339
12          12               5
13          13             279
14          14            1506
15          15             357
16          16             236
17          17             441
18          18               0
19          19             306
20          20              70
21          21             601
22          22             118
23          23              19
24          24               2
25          25              30
26          26              32
27          27               2
28          28              17
29          29               3
30          30               3
31      

In [10]:
top20 = department_counts.sort_values(
    by=["internal_edges", "department"],
    ascending=[False, True]
).head(20).reset_index(drop=True)

# Add zero-based ranking positions
top20.index.name = "rank"

print("TOP 20 DEPARTMENTS")
print(top20)

TOP 20 DEPARTMENTS
      department  internal_edges
rank                            
0             14            1506
1              4            1167
2              7             679
3             21             601
4              1             502
5             10             472
6             17             441
7              0             426
8             15             357
9             11             339
10            19             306
11            13             279
12            16             236
13            36             208
14             5             147
15             9             146
16             8             134
17            22             118
18            37              99
19            20              70


In [11]:
department_counts.to_csv(
    "all_42_department_counts.csv",
    index=False
)

top20.to_csv(
    "top20_departments.csv"
)

In [12]:
department_A = top20.iloc[4]["department"]
department_B = top20.iloc[14]["department"]

print("Department A:", department_A)
print("Department B:", department_B)

Department A: 1
Department B: 5


In [13]:
parameters = {
    "R": R,
    "pair_number": pair_number,
    "department_A": department_A,
    "department_B": department_B,
    "random_networks": num_random_networks,
    "swap_multiplier": swap_multiplier,
    "random_seed": random_seed
}

pd.DataFrame([parameters]).to_csv(
    "parameter_report.csv",
    index=False
)

In [14]:
nodes_A = departments[
    departments["department"] == department_A
]["node"].tolist()

nodes_B = departments[
    departments["department"] == department_B
]["node"].tolist()

print("Department A:", department_A)
print("Number of nodes in A:", len(nodes_A))

print("Department B:", department_B)
print("Number of nodes in B:", len(nodes_B))

Department A: 1
Number of nodes in A: 65
Department B: 5
Number of nodes in B: 18


In [15]:
edges_A = clean_edges[
    clean_edges["source"].isin(nodes_A) &
    clean_edges["target"].isin(nodes_A)
].copy()

edges_B = clean_edges[
    clean_edges["source"].isin(nodes_B) &
    clean_edges["target"].isin(nodes_B)
].copy()

print("Department A:", department_A)
print("Internal edges:", len(edges_A))

print("\nDepartment B:", department_B)
print("Internal edges:", len(edges_B))

Department A: 1
Internal edges: 502

Department B: 5
Internal edges: 147


In [16]:
pd.DataFrame({"node": nodes_A}).to_csv(
    "department_1_nodes.csv", index=False
)

edges_A.to_csv("department_1_edges.csv", index=False)

pd.DataFrame({"node": nodes_B}).to_csv(
    "department_5_nodes.csv", index=False
)

edges_B.to_csv("department_5_edges.csv", index=False)

print("All four files saved successfully!")

All four files saved successfully!
